# 05 — Prediction Pipeline
Wraps the saved ensemble into a `StockMovementPredictor` class.  
This is imported by the backtesting notebook (`06_backtesting.ipynb`).  
Also demonstrates single-headline and batch inference.

In [3]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import warnings
warnings.filterwarnings('ignore')

ROOT   = Path('..')
MODELS = ROOT / 'models' / 'saved_models'
PROC   = ROOT / 'data' / 'processed'

In [4]:
class StockMovementPredictor:
    """
    Wraps the trained ensemble for single or batch prediction.

    Usage
    -----
    predictor = StockMovementPredictor()
    result = predictor.predict('HDFC Bank Q3 net profit rises 18%', '2024-01-15', 'HDFCBANK')
    # → {'predicted_movement': 'UP', 'confidence': 0.74, 'probabilities': {...}}
    """

    BANKING_BOOSTS = {
        'npa': -1.5, 'fraud': -2.0, 'default': -1.5, 'downgrade': -1.8,
        'stressed': -1.2, 'recovery': +1.5, 'profit': +1.8, 'dividend': +1.5,
        'acquisition': +0.8, 'merger': +0.8, 'rally': +1.5, 'surge': +1.5,
        'crash': -2.0, 'collapse': -2.0, 'penalty': -1.5, 'fine': -1.0,
        'upgrade': +1.5, 'growth': +1.2, 'record': +1.0
    }

    def __init__(self, ensemble_path=None, verbose=True):
        if ensemble_path is None:
            ensemble_path = MODELS / 'ensemble_stock_predictor.pkl'

        with open(ensemble_path, 'rb') as f:
            bundle = pickle.load(f)

        self.rf            = bundle['rf']
        self.gb            = bundle['gb']
        self.meta_lr       = bundle['meta_lr']
        self.tfidf         = bundle['tfidf']
        self.scaler        = bundle['scaler']
        self.le            = bundle['le']
        self.tabular_cols  = bundle['tabular_cols']

        self.sia = SentimentIntensityAnalyzer()
        self.sia.lexicon.update(self.BANKING_BOOSTS)

        if verbose:
            print(f'✅ StockMovementPredictor loaded — classes: {list(self.le.classes_)}')

    def _build_row(self, headline, date_str, body=''):
        date = pd.to_datetime(date_str)
        vs   = self.sia.polarity_scores(headline)
        text = (headline + ' ' + body).strip()[:1500]
        return {
            'combined_text_512': text,
            'headline_len'     : len(headline.split()),
            'vader_pos'        : vs['pos'],
            'vader_neg'        : vs['neg'],
            'vader_neu'        : vs['neu'],
            'vader_compound'   : vs['compound'],
            'day_of_week'      : date.dayofweek,
            'month'            : date.month,
            'quarter'          : date.quarter,
            'is_monday'        : int(date.dayofweek == 0),
            'is_friday'        : int(date.dayofweek == 4),
            'is_month_end'     : int(date.is_month_end),
            'is_qtr_end'       : int(date.is_quarter_end),
        }

    def predict(self, headline, date_str, ticker=None, body='', verbose=True):
        import scipy.sparse as sp
        row = self._build_row(headline, date_str, body)

        X_tfidf = self.tfidf.transform([row['combined_text_512']])
        X_tab   = self.scaler.transform(
            pd.DataFrame([row])[self.tabular_cols].values
        )
        X = sp.hstack([X_tfidf, sp.csr_matrix(X_tab)])

        meta = np.hstack([
            self.rf.predict_proba(X),
            self.gb.predict_proba(X.toarray())
        ])
        proba     = self.meta_lr.predict_proba(meta)[0]
        pred_idx  = proba.argmax()
        pred_label = self.le.classes_[pred_idx]
        confidence = float(proba[pred_idx])

        result = {
            'predicted_movement': pred_label,
            'confidence'        : confidence,
            'probabilities'     : dict(zip(self.le.classes_, proba.round(4).tolist()))
        }
        if verbose:
            print(f'[{ticker or "?"}]  {pred_label}  ({confidence:.1%})  |  {result["probabilities"]}')
        return result

    def predict_batch(self, df, headline_col='headline', date_col='date',
                      ticker_col='ticker', body_col=None):
        results = []
        for _, row in df.iterrows():
            body = str(row[body_col]) if body_col and body_col in row else ''
            res  = self.predict(str(row[headline_col]), str(row[date_col]),
                                ticker=row.get(ticker_col), body=body, verbose=False)
            results.append(res)
        return pd.DataFrame(results)

print('StockMovementPredictor class defined.')

StockMovementPredictor class defined.


## Demo — single prediction

In [5]:
# Requires trained model from notebook 04
try:
    predictor = StockMovementPredictor()

    samples = [
        ('HDFC Bank Q3 net profit jumps 18%, asset quality improves', '2024-01-15', 'HDFCBANK'),
        ('SBI faces Rs 500 crore penalty over KYC violations', '2024-03-10', 'SBIN'),
        ('ICICI Bank announces Rs 12 per share dividend', '2024-04-22', 'ICICIBANK'),
        ('Axis Bank NPA rises sharply amid retail stress', '2024-02-05', 'AXISBANK'),
    ]

    for headline, date, ticker in samples:
        predictor.predict(headline, date, ticker)
except FileNotFoundError:
    print('⚠️  Run notebook 04 first to train and save the model.')

✅ StockMovementPredictor loaded — classes: ['DOWN', 'NEUTRAL', 'UP']
[HDFCBANK]  UP  (40.5%)  |  {'DOWN': 0.3985, 'NEUTRAL': 0.1967, 'UP': 0.4047}
[SBIN]  UP  (47.9%)  |  {'DOWN': 0.3096, 'NEUTRAL': 0.211, 'UP': 0.4794}
[ICICIBANK]  UP  (42.7%)  |  {'DOWN': 0.3577, 'NEUTRAL': 0.2155, 'UP': 0.4268}
[AXISBANK]  DOWN  (48.1%)  |  {'DOWN': 0.4812, 'NEUTRAL': 0.1633, 'UP': 0.3554}


## Demo — batch prediction on test set

In [6]:
try:
    features = pd.read_csv(PROC / 'features.csv', parse_dates=['date'])
    sample   = features.dropna(subset=['label_1d']).tail(50).copy()

    preds = predictor.predict_batch(sample, headline_col='headline',
                                    date_col='date', ticker_col='ticker',
                                    body_col='text')
    preds.index = sample.index
    comparison = pd.concat([sample[['ticker','date','headline','label_1d']], preds], axis=1)
    print(comparison.head(10).to_string())

    acc = (comparison['predicted_movement'] == comparison['label_1d']).mean()
    print(f'\nSample accuracy: {acc:.1%}')
except FileNotFoundError:
    print('⚠️  Run notebooks 03 and 04 first.')

      ticker       date                                                                                                                         headline label_1d predicted_movement  confidence                                      probabilities
13584   SBIN 2026-03-23                                             Sula Vineyards promoter Rajeev Samant buys Rs 3-crore shares, raises stake to 23.27%  NEUTRAL               DOWN    0.400605   {'DOWN': 0.4006, 'NEUTRAL': 0.2164, 'UP': 0.383}
13585   SBIN 2026-03-23                                                    Svatantra Microfin acquires Chaitanya India Fin; becomes 2nd largest NBFC-MFI  NEUTRAL                 UP    0.429899  {'DOWN': 0.3135, 'NEUTRAL': 0.2566, 'UP': 0.4299}
13586   SBIN 2026-03-23                                                    OpenClaw unveils ClawHub Marketplace; draws more than 331,000 stars on GitHub  NEUTRAL               DOWN    0.392541  {'DOWN': 0.3925, 'NEUTRAL': 0.2224, 'UP': 0.3851}
13587   SBIN 2026-03-23 